# Part 10: Full-Stack Deployment — Frontend + Backend to Cloud Run
## Content Creation Studio Workshop — Series Finale

Welcome to Part 10 of the Content Creation Studio Playbook — **the series finale!**

You've journeyed from your first agent to cloud deployment with Agent Engine. Now let's
complete the stack with a **full web application** that anyone can access from a browser.

---

## The Content Creation Studio Playbook Series
- Part 1: Building Your First AI Agent ✅
- Part 2: Giving Your Agent Custom Tools ✅
- Part 3: Building Agent Teams with Agent-as-a-Tool ✅
- Part 4: Sequential Workflows & Design Patterns ✅
- Part 5: Building Iterative Workflows with LoopAgent ✅
- Part 6: Efficient Workflows with ParallelAgent ✅
- Part 7: Callbacks, Context & Memory ✅
- Part 8: The Capstone Project ✅
- Part 9: Deploying to Vertex AI Agent Engine ✅
- **Part 10: Full-Stack Deployment with Cloud Run** (You are here) 👈

---

## What You'll Learn

In this notebook, you'll learn:
- **Full-Stack Architecture** — How React frontend + FastAPI backend + Agent Engine fit together
- **Backend-to-Agent Engine Integration** — Connecting your API server to your deployed agents
- **Docker Multi-Stage Builds** — Packaging frontend and backend into a single container
- **Cloud Run Deployment** — Serverless, auto-scaling hosting for your web app
- **IAM & Security** — Service accounts and permissions
- **Monitoring & Troubleshooting** — Logs, health checks, and common issues

---

## Prerequisites

Before starting, ensure you have:
1. Completed **Parts 1–9** of this workshop
2. An **Agent Engine endpoint** from Part 9 (`AGENT_ENGINE_RESOURCE_NAME`)
3. A Google Cloud project with **billing enabled**
4. **Docker** installed locally
5. **Node.js 20+** installed (for the frontend build)
6. Google Cloud SDK (`gcloud`) configured

---

## The Big Picture: End-to-End Architecture

Here's what we're building — a production web app accessible to anyone:

```
┌───────────────────────────────────────────────────────────────────────┐
│                    Cloud Run Service                           │
│                  (Serverless, Auto-scaling)                     │
│                                                                 │
│  ┌───────────────────┐     ┌──────────────────────────────┐  │
│  │  React Frontend  │     │  FastAPI Backend              │  │
│  │  (Static Files)  │     │                              │  │
│  │                   │────▶│  /api/create-content          │  │
│  │  - Content Form  │     │  /api/analyze-text            │  │
│  │  - Live Results  │◀─SSE│  /health                      │  │
│  │  - Multi-channel │     │                              │  │
│  └───────────────────┘     └──────────────┬───────────────┘  │
└──────────────────────────────────┬──────────────────────────────────┘
                                 │ gRPC
                                 ▼
                   ┌──────────────────────────────┐
                   │  Vertex AI Agent Engine       │
                   │  (From Part 9)                │
                   │                               │
                   │  11 Agents • 3 Workflows      │
                   │  Sequential • Loop • Parallel │
                   └─────────────┬────────────────┘
                                │
                                ▼
                        ┌──────────────┐
                        │  Gemini 2.5  │
                        └──────────────┘
```

**Key design decision:** We package both the React frontend and FastAPI backend into a
**single Cloud Run service**. The backend serves the frontend's static files and also
handles API requests — keeping the deployment simple with one URL for everything.

---

## 1. Backend: FastAPI + Agent Engine Connection

The backend (`backend/api_server.py`) is a FastAPI server that:
1. Connects to your **Agent Engine endpoint** from Part 9
2. Exposes REST API endpoints for the frontend
3. Streams responses back to the browser via **Server-Sent Events (SSE)**
4. Serves the React frontend as static files in production

### Connection Flow

```
1. User → POST /api/create-content
2. Backend → Connect to Agent Engine via AGENT_ENGINE_RESOURCE_NAME
3. Agent Engine → Execute the multi-agent workflow (11 agents)
4. Response → Stream back to browser via SSE
5. Frontend → Display results in real-time per channel
```

### Key Backend Code

Let's examine the core pieces of our backend.

In [ ]:
# backend/api_server.py — Core connection to Agent Engine
#
# This is a walkthrough of the actual code in your project.
# The real file is at: content_creation_mas/backend/api_server.py

# --- 1. Connect to Agent Engine ---
# The AGENT_RESOURCE_NAME comes from Part 9's deployment

import os
from vertexai import agent_engines

AGENT_RESOURCE_NAME = os.environ.get("AGENT_ENGINE_RESOURCE_NAME")

# This single line connects to your deployed 11-agent system!
remote_agent = agent_engines.get(AGENT_RESOURCE_NAME)

print("This cell shows the code pattern — it won't run without a deployed agent.")
print("The actual backend code lives in: backend/api_server.py")

### API Endpoints

The backend exposes three endpoints:

| Endpoint | Method | Purpose |
|---|---|---|
| `/api/create-content` | POST | Full content creation workflow (SSE streaming) |
| `/api/analyze-text` | POST | Quick text analysis (word count, readability, hashtags) |
| `/health` | GET | Health check for monitoring |

### Streaming with Server-Sent Events (SSE)

The content creation endpoint uses SSE to stream results in real-time.
Each parallel agent's output arrives as a separate **channel**:

In [ ]:
# How the backend maps agent outputs to frontend channels
#
# With sub_agents, inner events propagate and include the author field.
# The backend maps each author to a named channel for the frontend.

CHANNEL_MAP = {
    "blog_post_writer_agent": "blog_post",
    "social_media_creator_agent": "social_media",
    "email_newsletter_writer_agent": "email_newsletter",
    "seo_metadata_agent": "seo_metadata",
}

# When the backend receives an event from Agent Engine:
#   1. Check the event's 'author' field
#   2. Map it to a channel name
#   3. Send it to the frontend as an SSE event:
#      data: {"type": "content_piece", "channel": "blog_post", "content": "..."}

print("Channel mapping: agent author → frontend channel")
for agent, channel in CHANNEL_MAP.items():
    print(f"  {agent} → {channel}")

### The Streaming Endpoint

Here's the core of the `/api/create-content` endpoint — it streams Agent Engine events
to the browser as they arrive:

In [ ]:
# Simplified version of the streaming logic in api_server.py
#
# The actual code handles error cases, session management,
# and proper SSE formatting. See backend/api_server.py for the full version.

async def generate_content_stream(remote_agent, query, user_id, session_id):
    """Stream content creation events from Agent Engine to the frontend."""
    import json

    async for event in remote_agent.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message=query
    ):
        if not isinstance(event, dict):
            continue

        # Extract author and text
        author = event.get("author", "")
        channel = CHANNEL_MAP.get(author)

        text = ""
        content = event.get("content", {})
        if isinstance(content, dict):
            for part in content.get("parts", []):
                if isinstance(part, dict) and part.get("text"):
                    text += part["text"]

        # Send channel-specific content to frontend
        if channel and text:
            yield f"data: {json.dumps({'type': 'content_piece', 'channel': channel, 'content': text})}\n\n"

    yield f"data: {json.dumps({'type': 'complete'})}\n\n"

print("This shows the streaming pattern used in api_server.py")
print("The frontend receives events like:")
print('  data: {"type": "content_piece", "channel": "blog_post", "content": "..."}')
print('  data: {"type": "content_piece", "channel": "social_media", "content": "..."}')
print('  data: {"type": "complete"}')

---

## 2. Docker: Multi-Stage Build

We package the React frontend and Python backend into a **single Docker container**
using a multi-stage build. This keeps the image small and the deployment simple.

### How It Works

```
Stage 1: Node.js              Stage 2: Python
┌────────────────────┐     ┌────────────────────┐
│  npm ci             │     │  pip install        │
│  npm run build      │───▶│  COPY dist → static  │
│  → /frontend/dist  │     │  COPY api_server.py │
└────────────────────┘     │  uvicorn :8080      │
  (discarded)              └────────────────────┘
                             (final image)
```

**Stage 1** builds the React app into static HTML/JS/CSS.
**Stage 2** installs Python dependencies, copies the built frontend, and runs uvicorn.
The Node.js layer is discarded — only the built output survives.

In [ ]:
# Our Dockerfile (content_creation_mas/Dockerfile)
#
# This is the actual Dockerfile used for deployment.

dockerfile_content = """
# Stage 1: Build React frontend
FROM node:20-alpine AS frontend-build
WORKDIR /frontend
COPY frontend/package*.json ./
RUN npm ci
COPY frontend/ ./
RUN rm -f .env .env.local .env.development .env.production
RUN npm run build

# Stage 2: Python backend + built frontend
FROM python:3.11-slim
WORKDIR /app
RUN apt-get update && apt-get install -y gcc && rm -rf /var/lib/apt/lists/*
COPY backend/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY backend/api_server.py ./
COPY --from=frontend-build /frontend/dist ./static

ENV PORT=8080
EXPOSE 8080
CMD exec uvicorn api_server:app --host 0.0.0.0 --port ${PORT}
"""

print("Dockerfile walkthrough:")
print("  Stage 1: Builds React frontend into /frontend/dist")
print("  Stage 2: Installs Python deps, copies built frontend as static files")
print("  Result:  Single container serving both frontend and API on port 8080")

---

## 3. Deployment: Cloud Run

Cloud Run is Google's serverless container platform. It auto-scales from 0 to N instances
based on traffic — you only pay when requests are being processed.

### The Deployment Script

The `deployment/deploy-combined.sh` script automates the entire process:

```
Step 1/5: Configure Docker authentication with Artifact Registry
Step 2/5: Build Docker image (frontend + backend)
Step 3/5: Push image to Artifact Registry
Step 4/5: Deploy to Cloud Run with environment variables
Step 5/5: Retrieve and display service URL
```

### Prerequisites: Ensure Agent Engine Is Deployed

Your `.env` file should have the `AGENT_ENGINE_RESOURCE_NAME` from Part 9:

In [ ]:
# Step 1: Verify your .env file has the required variables

import os
from pathlib import Path

env_path = Path("../.env")
if env_path.exists():
    print("Found .env file. Checking required variables...\n")
    with open(env_path) as f:
        env_content = f.read()
    
    required_vars = [
        "GOOGLE_CLOUD_PROJECT",
        "GOOGLE_CLOUD_LOCATION",
        "AGENT_ENGINE_RESOURCE_NAME",
    ]
    for var in required_vars:
        if var in env_content:
            print(f"  {var}: Found")
        else:
            print(f"  {var}: MISSING")
    
    print("\nIf AGENT_ENGINE_RESOURCE_NAME is missing, deploy your agent first (Part 9).")
else:
    print("No .env file found at project root.")
    print("Create one with:")
    print('  GOOGLE_CLOUD_PROJECT=your-project-id')
    print('  GOOGLE_CLOUD_LOCATION=us-central1')
    print('  AGENT_ENGINE_RESOURCE_NAME=projects/.../reasoningEngines/...')

### Deploy to Cloud Run

Run the deployment script from your terminal:

```bash
cd content_creation_mas/deployment
./deploy-combined.sh
```

**What it does:**

1. Loads your `.env` configuration
2. Authenticates Docker with Artifact Registry
3. Builds the multi-stage Docker image (React + FastAPI)
4. Pushes the image to `{REGION}-docker.pkg.dev/{PROJECT}/content-studio/`
5. Deploys to Cloud Run with:
   - 2 vCPU, 2 GB memory
   - 5-minute timeout (for long content workflows)
   - Auto-scaling 0–10 instances
   - Service account: `content-studio-sa@{PROJECT}.iam.gserviceaccount.com`
   - Environment variables: `AGENT_RESOURCE_NAME`, `GOOGLE_CLOUD_PROJECT`, etc.
6. Returns your live service URL

In [ ]:
# You can also trigger the deployment from this notebook
# (Uncomment to run — this will take several minutes)

# !cd .. && bash deployment/deploy-combined.sh

---

## 4. IAM & Security

Cloud Run uses a **service account** to authenticate with Agent Engine. The `setup_gcp.sh`
script from Part 9 already created this, but here's what's required:

### Required IAM Roles

| Role | Purpose |
|---|---|
| `roles/aiplatform.user` | Access Agent Engine endpoints |
| `roles/run.invoker` | Invoke Cloud Run services |
| `roles/storage.objectViewer` | Read from Cloud Storage (staging) |
| `roles/logging.logWriter` | Write logs to Cloud Logging |

In [ ]:
# Verify IAM setup for the service account
# (Uncomment to run)

# import os
# PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT', 'your-project-id')
# SERVICE_ACCOUNT = f"content-studio-sa@{PROJECT_ID}.iam.gserviceaccount.com"

# !gcloud projects get-iam-policy {PROJECT_ID} \
#     --flatten="bindings[].members" \
#     --filter="bindings.members:serviceAccount:{SERVICE_ACCOUNT}" \
#     --format="table(bindings.role)"

print("Service account: content-studio-sa@{PROJECT}.iam.gserviceaccount.com")
print("\nRequired roles:")
print("  roles/aiplatform.user      → Access Agent Engine")
print("  roles/run.invoker          → Invoke Cloud Run")
print("  roles/storage.objectViewer → Read Cloud Storage")
print("  roles/logging.logWriter    → Write logs")

---

## 5. Testing the Production Deployment

Once deployed, you can test your live service.

In [ ]:
# Step 1: Get your Cloud Run service URL
# (Uncomment to run)

# !gcloud run services describe content-studio \
#     --region=us-central1 \
#     --format='value(status.url)'

print("To get your service URL, run:")
print("  gcloud run services describe content-studio --region=us-central1 --format='value(status.url)'")

In [ ]:
# Step 2: Test the health endpoint
# Replace YOUR_URL with your actual Cloud Run URL
# (Uncomment to run)

# !curl -s https://YOUR_URL/health | python3 -m json.tool

# Expected response:
# {
#     "status": "ok",
#     "agent": "content_creation_studio",
#     "agent_resource": "projects/.../reasoningEngines/...",
#     "agent_connected": true
# }

print("Health check verifies:")
print("  1. Backend is running")
print("  2. Agent Engine is configured")
print("  3. Connection to Agent Engine is active")

In [ ]:
# Step 3: Test content creation via API
# Replace YOUR_URL with your actual Cloud Run URL
# (Uncomment to run)

# !curl -X POST "https://YOUR_URL/api/create-content" \
#     -H "Content-Type: application/json" \
#     -d '{
#         "topic": "AI in Healthcare",
#         "target_audience": "Healthcare professionals",
#         "tone": "Professional",
#         "keywords": "AI, healthcare, diagnostics"
#     }'

print("The API returns an SSE stream with events like:")
print('  data: {"type": "status", "message": "Starting content creation..."}' )
print('  data: {"type": "content_piece", "channel": "blog_post", "content": "..."}' )
print('  data: {"type": "content_piece", "channel": "social_media", "content": "..."}' )
print('  data: {"type": "content_piece", "channel": "email_newsletter", "content": "..."}' )
print('  data: {"type": "content_piece", "channel": "seo_metadata", "content": "..."}' )
print('  data: {"type": "complete"}')

---

## 6. Monitoring & Logs

Cloud Run integrates with Google Cloud's observability stack out of the box.

In [ ]:
# Cloud Run Logs — see what your web app is doing
# (Uncomment to run)

# Real-time logs:
# !gcloud run services logs read content-studio --region=us-central1 --limit=20

# Filter for errors only:
# !gcloud run services logs read content-studio --region=us-central1 --filter="severity>=ERROR"

# Agent Engine logs (from Part 9):
# !gcloud logging read "resource.type=aiplatform.googleapis.com/ReasoningEngine" --limit=10

print("Monitoring commands:")
print("  Cloud Run logs:    gcloud run services logs read content-studio --region=us-central1")
print("  Errors only:       ... --filter=\"severity>=ERROR\"")
print("  Agent Engine logs: gcloud logging read \"resource.type=aiplatform.googleapis.com/ReasoningEngine\"")

---

## 7. Troubleshooting

### Common Issues and Fixes

#### Issue 1: `AGENT_ENGINE_RESOURCE_NAME not set`

```bash
# Check environment variables on Cloud Run
gcloud run services describe content-studio \
    --region=us-central1 \
    --format="value(spec.template.spec.containers[0].env)"

# Fix: update the env var
gcloud run services update content-studio \
    --region=us-central1 \
    --set-env-vars="AGENT_RESOURCE_NAME=projects/.../reasoningEngines/..."
```

#### Issue 2: `Permission Denied`

```bash
# Check the service account
gcloud run services describe content-studio \
    --region=us-central1 \
    --format="value(spec.template.spec.serviceAccountName)"

# Add the required role
gcloud projects add-iam-policy-binding $PROJECT_ID \
    --member="serviceAccount:content-studio-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
    --role="roles/aiplatform.user"
```

#### Issue 3: `Connection Timeout`

Content creation workflows can take several minutes. Increase the Cloud Run timeout:

```bash
gcloud run services update content-studio \
    --region=us-central1 \
    --timeout=300
```

#### Issue 4: `Agent Engine Not Found`

- Verify the agent is deployed: `gcloud beta ai reasoning-engines list --location=us-central1`
- Check the resource name matches exactly
- Ensure Agent Engine and Cloud Run are in the **same region**

---

## 8. Deployment Best Practices

**Note:** These are basic best practices for learning and demonstration deployments.
Production systems require significantly more robust practices.

### Region Consistency

Keep all services in the same region to minimize latency:

```bash
REGION="us-central1"  # Agent Engine + Cloud Run + Artifact Registry
```

### Error Handling

In [ ]:
# Best practice: wrap Agent Engine calls with proper error handling

import google.api_core.exceptions
from google.api_core.retry import Retry

# --- Retry Logic ---
# Transient errors (network blips, rate limits) are automatically retried
retry_config = Retry(
    initial=1.0,       # Wait 1s before first retry
    maximum=10.0,      # Max wait between retries
    multiplier=2.0,    # Double the wait each retry
    deadline=60.0      # Give up after 60s total
)

# --- Error Handling ---
# Always catch Agent Engine errors and return meaningful HTTP responses
try:
    # response = agent.query(user_query, retry=retry_config)
    print("Agent query would execute here with retry logic")
except google.api_core.exceptions.GoogleAPIError as e:
    print(f"Agent Engine error: {e}")
    # In FastAPI: raise HTTPException(status_code=500, detail="Agent unavailable")

print("\nBest practices demonstrated:")
print("  1. Exponential backoff retry for transient errors")
print("  2. Deadline to prevent indefinite hangs")
print("  3. Specific exception handling with clear error messages")

---

## 9. Cleanup (Optional)

When you're done testing, clean up to avoid ongoing charges.

In [ ]:
# Cleanup commands (uncomment to run)

# Delete Cloud Run service:
# !gcloud run services delete content-studio --region=us-central1 --quiet

# Delete Agent Engine deployment (from Part 9):
# AGENT_ID = "your-agent-id"  # Last part of AGENT_ENGINE_RESOURCE_NAME
# !gcloud beta ai reasoning-engines delete {AGENT_ID} --location=us-central1 --quiet

# Delete Docker images from Artifact Registry:
# !gcloud artifacts docker images delete \
#     us-central1-docker.pkg.dev/{PROJECT_ID}/content-studio/content-studio --quiet

print("Cleanup commands are commented out for safety.")
print("Uncomment and run only when you're sure you want to delete.")
print("\nSee deployment/CLEANUP_GUIDE.md for a comprehensive cleanup guide.")

---

## Complete Workshop Recap: Parts 1–10

### The Full Journey

| Part | Topic | What You Built |
|------|-------|----------------|
| 1 | First Agent | Single agent with Google Search |
| 2 | Custom Tools | Specialist agent with Python function tools |
| 3 | Agent Teams | Orchestrator managing specialists via AgentTool |
| 4 | Sequential Workflows | Ordered pipelines with SequentialAgent |
| 5 | Iterative Workflows | Quality loops with LoopAgent |
| 6 | Parallel Workflows | Concurrent execution with ParallelAgent |
| 7 | Callbacks & Memory | Runtime control, state, artifacts, cross-session memory |
| 8 | Capstone Project | Complete 11-agent autonomous system |
| 9 | Agent Engine | Cloud deployment of your agent system |
| **10** | **Cloud Run** | **Full-stack web app accessible to anyone** |

### The Complete Architecture Stack

```
Layer 5: Web Application (Part 10)
  React Frontend + FastAPI Backend on Cloud Run
          │
Layer 4: Cloud Infrastructure (Part 9)
  Vertex AI Agent Engine (managed, auto-scaling)
          │
Layer 3: Orchestration (Parts 3, 8)
  Master Orchestrator + Callbacks + Memory
          │
Layer 2: Workflows (Parts 4–6)
  Sequential → Loop → Parallel
          │
Layer 1: Agents & Tools (Parts 1–2, 7)
  11 Specialist Agents + Custom Tools + State
```

---

## Congratulations — You've Completed the Series!

You've gone from building a single AI agent to deploying a **full-stack, multi-agent
web application** on Google Cloud.

### What You Can Now Do

- Build multi-agent systems with Google ADK
- Design complex workflows (sequential, iterative, parallel)
- Add runtime control with callbacks and cross-session memory
- Deploy agents to Vertex AI Agent Engine
- Build and deploy full-stack web apps on Cloud Run
- Monitor and troubleshoot production AI systems

### Next Steps

1. **Customize** — Adapt the Content Creation Studio for your own domain
2. **Extend** — Add new specialist agents, tools, or workflow patterns
3. **Harden** — Add authentication, CI/CD, comprehensive testing for production
4. **Share** — Build something amazing and share your learnings!

### Resources

- [Google ADK Documentation](https://google.github.io/adk-docs/)
- [Vertex AI Agent Engine](https://cloud.google.com/vertex-ai/docs/agent-engine)
- [Cloud Run Documentation](https://cloud.google.com/run/docs)
- [GitHub Repository](https://github.com/Saoussen-CH/content_creation_mas_workshop)

---

**Thank you for completing the Google ADK Content Creation Studio Playbook!**

**End of Part 10 — Series Complete**